<a href="https://colab.research.google.com/github/dolunay38/BookVoice-AI/blob/main/BookVoice_AI_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ BookVoice-AI — GPU Studio
**KI Hörbuch mit NVIDIA T4 GPU (kostenlos)**

## Schnellstart
1. Laufzeit → Laufzeittyp ändern → **T4 GPU**
2. **Strg+F9** (Alle ausführen)
3. Warten bis URL erscheint (~10 Min beim ersten Mal)
4. URL in BookVoice-AI GUI eingeben → GPU Colab → Verbinden

> Session läuft ~3-4 Stunden. Danach nur Zelle 2 neu ausführen.

In [28]:
# ── ZELLE 1: Alles installieren (nur beim ersten Mal) ──
import subprocess, sys

print('Schritt 1: System-Pakete...')
subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg', 'calibre', 'tesseract-ocr', 'tesseract-ocr-tur', 'tesseract-ocr-deu'], capture_output=True)
print('OK: System-Pakete!')

print('Schritt 2: Python-Pakete...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'coqui-tts', 'fastapi', 'uvicorn', 'python-multipart', 'pytesseract', 'pdf2image', 'PyPDF2', 'pillow', 'ebooklib', 'edge-tts'], capture_output=True)
print('OK: Python-Pakete!')

print('Schritt 3: tts_server.py laden...')
import urllib.request
urllib.request.urlretrieve('https://raw.githubusercontent.com/dolunay38/BookVoice-AI/main/tts_server.py', 'tts_server.py')
print('OK: tts_server.py!')

print('Schritt 4: Ordner anlegen...')
import os
for d in ['/content/HOERBUCH', '/content/tts_models', '/content/musik', '/content/covers']:
    os.makedirs(d, exist_ok=True)
print('OK: Ordner!')

print('Schritt 5: XTTS-v2 Modell laden (~1.8 GB)...')
os.environ['COQUI_TOS_AGREED'] = '1'
from TTS.api import TTS
TTS('tts_models/multilingual/multi-dataset/xtts_v2')
print('OK: Modell bereit!')

print('Schritt 6: Standard-Stimme...')
import numpy as np, torch, torchaudio
voice_path = '/content/tts_models/stimme.wav'
if not os.path.exists(voice_path):
    t = np.linspace(0, 3, 22050*3)
    audio = torch.tensor(np.sin(2*np.pi*200*t)*0.3, dtype=torch.float32).unsqueeze(0)
    torchaudio.save(voice_path, audio, 22050)
print('OK: Stimme!')

print('Schritt 7: Cloudflared...')
subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/usr/local/bin/cloudflared'], capture_output=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])
print('OK: Cloudflared!')

print('\n=== INSTALLATION FERTIG ===')

Schritt 1: System-Pakete...
OK: System-Pakete!
Schritt 2: Python-Pakete...
OK: Python-Pakete!
Schritt 3: tts_server.py laden...
OK: tts_server.py!
Schritt 4: Ordner anlegen...
OK: Ordner!
Schritt 5: XTTS-v2 Modell laden (~1.8 GB)...


100%|██████████| 1.87G/1.87G [00:23<00:00, 80.5MiB/s]
4.37kiB [00:00, 5.72MiB/s]
361kiB [00:00, 99.2MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 52.8kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 93.3MiB/s]


OK: Modell bereit!
Schritt 6: Standard-Stimme...
OK: Stimme!
Schritt 7: Cloudflared...
OK: Cloudflared!

=== INSTALLATION FERTIG ===


In [29]:
# ── ZELLE 2: Server starten (bei jedem Neustart) ──
import subprocess, os, time, re

# Alte Prozesse beenden
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# Server starten
env = os.environ.copy()
env['TTS_OUTPUT'] = '/content/HOERBUCH'
env['TTS_LANG'] = 'tr'
env['COQUI_TOS_AGREED'] = '1'

server = subprocess.Popen(
    ['uvicorn', 'tts_server:app', '--host', '0.0.0.0', '--port', '7500'],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print('Server startet...')
time.sleep(8)

# Tunnel starten
proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:7500'],
    stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
)

print('Warte auf URL...')
for i in range(60):
    line = proc.stderr.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            print('='*50)
            print('BOOKVOICE-AI GPU BEREIT!')
            print(f'URL: {url}')
            print('='*50)
            print('1. URL kopieren')
            print('2. BookVoice-AI GUI oeffnen')
            print('3. GPU Colab Tab klicken')
            print('4. URL einfuegen -> Verbinden')
            break
    time.sleep(1)


Server startet...
Warte auf URL...
BOOKVOICE-AI GPU BEREIT!
URL: https://proxy-brad-san-operated.trycloudflare.com
1. URL kopieren
2. BookVoice-AI GUI oeffnen
3. GPU Colab Tab klicken
4. URL einfuegen -> Verbinden


In [ ]:
# ── ZELLE 3: Session aktiv halten ──
# Diese Zelle laufen lassen damit Colab nicht trennt!
import time
print('Session laeuft... (Diese Zelle aktiv lassen!)')
i = 0
while True:
    i += 1
    if i % 20 == 0:
        print(f'Aktiv seit {i//2} Minuten...')
    time.sleep(30)

Session laeuft... (Diese Zelle aktiv lassen!)


In [ ]:
# ── ZELLE 4: Eigene Stimme hochladen (optional) ──
from google.colab import files
print('Stimme hochladen (WAV/MP3, 30-60 Sek):')
uploaded = files.upload()
for filename, data in uploaded.items():
    with open(f'/content/tts_models/{filename}', 'wb') as f:
        f.write(data)
    print(f'Stimme gespeichert: {filename}')

In [ ]:
# ── ZELLE 5: Hoerbucher herunterladen (optional) ──
from google.colab import files
import glob
hoerbucher = glob.glob('/content/HOERBUCH/**/*.mp3', recursive=True) + glob.glob('/content/HOERBUCH/**/*.m4b', recursive=True)
if hoerbucher:
    print(f'{len(hoerbucher)} Hoerbuch gefunden:')
    for h in hoerbucher:
        print(f'  {h}')
        files.download(h)
else:
    print('Noch keine Hoerbucher generiert.')